# Batch HEALPix Map Visualization for Galaxy Properties

This notebook contains the complete, reproducible pipeline for visualizing precomputed HEALPix map data (`.npz` format). It processes the integrated maps across multiple distance thresholds ($10 \text{ to } 200 \text{ Mpc}$) and angular resolutions ($\text{NSIDE} = 1 \text{ to } 256$), generating publication-ready, all-sky maps using the Mollweide projection.

## Computational Environment & Dependencies

To ensure absolute scientific reproducibility and correct rendering of astronomical coordinate grids, this notebook requires standard data science packages and the specialized `healpy` projection toolkit. The exact versions used during development are pinned in the root directory's `requirements.txt`.

### Core Packages & Visualization Libraries:
* **Python Version:** `3.10` or higher (recommended)
* **healpy (v1.16.x or higher):** Python wrapper for HEALPix. This notebook extensively utilizes the advanced `healpy.newvisufunc.projview` interface to handle high-resolution spherical map rendering, graticule labels, and dynamic color-scaling.
* **numpy:** Used for reading `.npz` compressed binary arrays and masking empty pixels.
* **matplotlib:** Backend engine for figure rendering and exporting to vector graphics formats (`.pdf`).

### Visualization & Astrophysics Logic Notes:
1. **Empty Pixel Masking:** Pixels with no galaxy data (originally filled with a baseline value of $10^{-10}$) are dynamically replaced with the standard `hp.UNSEEN` flag. This ensures they are treated as masked regions and rendered transparently or colorbar-independent.
2. **Astronomical Projection:** The pipeline implements the **Mollweide projection** (an equal-area, pseudocylindrical map projection) with an astronomical orientation (`flip='astro'`), which is the standard convention for displaying celestial sphere distributions (e.g., cosmic microwave background, galaxy surveys) in major journals.
3. **Dynamic Contrast Scaling:** To prevent local high-density regions from washing out features, the colorbar limits are automatically bounded to 3 orders of magnitude below the maximum detected pixel value ($\text{max} - 3 \text{ dex}$).

### Environment Reproduction:
You can quickly set up your local or server environment to match this notebook using:
```bash
pip install -r requirements.txt

In [ ]:
"""
Batch HEALPix Map Visualization for Stellar Mass
Workflow:
    1. Load precomputed HEALPix map data (NPZ binary files)
    2. Replace empty pixel value (1e-10) with healpy standard UNSEEN flag
    3. Dynamically set color scale (restrict to 3 magnitude range)
    4. Generate Mollweide projection sky map
    5. Export figures to PDF format with auto directory creation

HEALPix Angular Resolution Lookup:
    NSIDE  ->  Pixel Angular Size

Projection: Mollweide (Astronomy standard all-sky projection)
Unit: log10(Stellar Mass / Msun)
"""
import os
import warnings
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from healpy.newvisufunc import projview

# -------------------------- Global Configuration & Constants --------------------------
# Suppress non-critical runtime warnings
warnings.filterwarnings("ignore")

# NSIDE to angular resolution label mapping (consistent with calculation scripts)
NSIDE_RES_MAPPING = {
    1: '58.6deg',
    2: '29.3deg',
    4: '14.7deg',
    8: '7.33deg',
    16: '3.66deg',
    32: '1.83deg',
    64: '55arcmin',
    128: '27.5arcmin',
    256: '13.7arcmin'
    # 512: '6.87arcmin',
    # 1024: '3.44arcmin',
    # 2048: '1.72arcmin'
}

# Target HEALPix NSIDE list for batch plotting
TARGET_NSIDES = [1, 2, 4, 8, 16, 32, 64, 128, 256]

# Distance range: 10 ~ 200 Mpc, step = 10 Mpc
DISTANCE_RANGE = range(10, 210, 10)


# Fixed computational & plotting constants
EMPTY_PIXEL_VALUE = 1e-10    # Original fill value for empty pixels
COLOR_SCALE_OFFSET = 3       # Restrict color range to 3 orders of magnitude
PLOT_DPI = 300               # Output figure DPI
FIG_WIDTH = 14               # Figure width for projview
GRID_SPACING = 30            # Sky grid spacing (deg)

# File paths (modify according to your environment)
DATA_ROOT_DIR = "./mapdata/MASS"    # Input HEALPix NPZ data
FIGURE_SAVE_DIR = "./figure/MASS"  # Output PDF figures

# -------------------------- Core Function Definition --------------------------
def plot_and_export_mass_sky_map(
    res_label: str,
    distance_cut: int
) -> None:
    """
    Load precomputed stellar mass HEALPix map from NPZ file, generate sky plot and save as PDF.

    Parameters
    ----------
    res_label : str
        Angular resolution label, used for subdirectory and filename matching
    distance_cut : int
        Distance threshold (Mpc) for the corresponding galaxy sample

    Data Format (NPZ):
        npix: Pixel index array
        ra: Right Ascension of pixel centers (deg)
        dec: Declination of pixel centers (deg)
        m: Logarithmic stellar mass for each pixel (main data for plotting)

    Data Processing Rules
    ---------------------
    1. Replace empty pixel value (1e-10) with hp.UNSEEN for blank display
    2. Set color bar range: fix to 3 orders of magnitude below maximum value
    3. If no valid data exists, set color range to [0, -3] by default

    Output
    ------
    High-resolution PDF figure saved to: {FIGURE_SAVE_DIR}/{res_label}/
    """
    # Build full path of input HEALPix NPZ data file (modified: .txt -> .npz)
    data_filename = f"MASS_D{distance_cut}_{res_label}.npz"
    data_path = os.path.join(DATA_ROOT_DIR, res_label, data_filename)

    # Check if input file exists
    if not os.path.isfile(data_path):
        raise FileNotFoundError(f"Missing data file: {data_path}")

    # -------------------------- 核心修改：读取 NPZ 格式 --------------------------
    # Load NPZ binary file & extract plotting data (key = 'value')
    npz_data = np.load(data_path)
    mass_map = npz_data['value'].copy()

    # Replace default empty value with healpy standard UNSEEN flag
    mass_map[mass_map == EMPTY_PIXEL_VALUE] = hp.UNSEEN

    # Calculate valid pixel values and determine color scale limits
    valid_data = mass_map[mass_map != hp.UNSEEN]
    if len(valid_data) > 0:
        val_max = np.max(valid_data)
        val_min = np.min(valid_data)
        # Restrict color range to 3 orders of magnitude
        if val_max - COLOR_SCALE_OFFSET > val_min:
            val_min = val_max - COLOR_SCALE_OFFSET
    else:
        # Fallback range when all pixels are empty
        val_max, val_min = 0, -3

    # Generate Mollweide all-sky projection plot
    projview(
        mass_map,
        coord=['C'],
        flip='astro',
        projection_type='mollweide',
        graticule=True,
        graticule_labels=True,
        nest=False,
        unit=r'log$(M/ \rm M_{\odot}$)',
        xlabel='RA',
        ylabel='DEC',
        cb_orientation='horizontal',
        min=val_min,
        max=val_max,
        latitude_grid_spacing=GRID_SPACING,
        longitude_grid_spacing=GRID_SPACING,
        fontsize={
            'xlabel': 21, 'ylabel': 21,
            'xtick_label': 20, 'ytick_label': 30,
            'cbar_label': 30, 'cbar_tick_label': 30
        },
        width=FIG_WIDTH,
        cmap='binary'
    )

    # Create output directory automatically
    fig_subdir = os.path.join(FIGURE_SAVE_DIR, res_label)
    os.makedirs(fig_subdir, exist_ok=True)

    # Save figure and close canvas to release memory
    fig_filename = f"MASS_D{distance_cut}_{res_label}.pdf"
    fig_save_path = os.path.join(fig_subdir, fig_filename)
    plt.savefig(fig_save_path, dpi=PLOT_DPI, format="pdf", bbox_inches='tight')
    plt.close()

    # Print progress information
    print(
        f"Finished | Resolution={res_label:12s} | Distance={distance_cut:3d} Mpc "
        f"| Saved to: {fig_save_path}"
    )

# -------------------------- Main Program Entry --------------------------
def main():
    print("=" * 60)
    print("Start Batch Visualization for Stellar Mass HEALPix Maps")
    print("=" * 60)

    # Batch loop over all target HEALPix resolutions
    for nside in TARGET_NSIDES:
        current_res = NSIDE_RES_MAPPING[nside]
        print(f"\n----- Processing Resolution: {current_res} (NSIDE = {nside}) -----")

        # Loop over all distance thresholds
        for dist in DISTANCE_RANGE:
            plot_and_export_mass_sky_map(current_res, dist)

    # Final log
    print("\n" + "=" * 60)
    print("All batch plotting tasks completed successfully.")
    print(f"All figures saved to: {FIGURE_SAVE_DIR}")
    print("=" * 60)

# Standard entry point for standalone execution or module import
if __name__ == "__main__":
    main()

In [ ]:
"""
Batch HEALPix Map Visualization for Star Formation Rate (SFR)
Workflow:
    1. Load precomputed HEALPix map data from NPZ binary files
    2. Replace empty pixel value (1e-10) with healpy standard UNSEEN flag
    3. Calculate color scale limits (restrict to 3 orders of magnitude)
    4. Generate all-sky map with Mollweide projection
    5. Export high-resolution figures to PDF and auto-create directories

HEALPix Angular Resolution Lookup:
    NSIDE  ->  Pixel Angular Size

Plot Settings:
    Projection: Mollweide (Astronomy standard all-sky projection)
    Colormap: binary
    Unit: log(SFR / Msun/yr)
"""
import os
import warnings
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from healpy.newvisufunc import projview

# -------------------------- Global Configuration & Constants --------------------------
# Suppress non-critical runtime warnings
warnings.filterwarnings("ignore")

# Mapping: HEALPix NSIDE -> Angular resolution label (consistent with calculation scripts)
NSIDE_RES_MAPPING = {
    1: '58.6deg',
    2: '29.3deg',
    4: '14.7deg',
    8: '7.33deg',
    16: '3.66deg',
    32: '1.83deg',
    64: '55arcmin',
    128: '27.5arcmin',
    256: '13.7arcmin'
    # 512: '6.87arcmin',
    # 1024: '3.44arcmin',
    # 2048: '1.72arcmin'
}

# Target HEALPix NSIDE list for batch plotting
TARGET_NSIDES = [1, 2, 4, 8, 16, 32, 64, 128, 256]

# Distance threshold range: 10 ~ 200 Mpc, step = 10 Mpc
DISTANCE_RANGE = range(10, 210, 10)

# Fixed computation & plotting constants
EMPTY_PIXEL_VALUE = 1e-10    # Default fill value for empty HEALPix pixels
COLOR_SCALE_OFFSET = 3       # Limit color range to 3 orders of magnitude
PLOT_DPI = 300               # Output figure resolution
FIG_WIDTH = 14               # Canvas width for sky map
GRID_SPACING = 30            # Longitude/latitude grid spacing (deg)

# File paths (modify according to your local/server environment)
DATA_ROOT_DIR = "./mapdata/SFR"        # Input HEALPix NPZ data
FIGURE_SAVE_DIR = "./figure/SFR"  # Output PDF figures
# =====================================================================================

# -------------------------- Core Function Definition --------------------------
def plot_and_export_sfr_sky_map(
    res_label: str,
    distance_cut: int
) -> None:
    """
    Load precomputed SFR HEALPix map from NPZ file, generate all-sky plot and save as PDF file.

    Parameters
    ----------
    res_label : str
        Angular resolution label, used for subdirectory and filename matching
    distance_cut : int
        Distance threshold (Mpc) of the corresponding galaxy sample

    Data Format (NPZ):
        npix: Pixel index array
        ra: Right Ascension of pixel centers (deg)
        dec: Declination of pixel centers (deg)
        m: Logarithmic SFR for each pixel (main data for plotting)

    Data Processing Rules
    ---------------------
    1. Replace empty pixel value (1e-10) with hp.UNSEEN for blank visualization
    2. Dynamically set color bar range: max value down to 3 orders of magnitude
    3. Fallback color range [0, -3] if no valid data exists

    Output
    ------
    High-resolution PDF figure saved to: {FIGURE_SAVE_DIR}/{res_label}/
    """
    # Construct full path of input NPZ data file (modified: .txt -> .npz)
    data_filename = f"SFR_D{distance_cut}_{res_label}.npz"
    data_path = os.path.join(DATA_ROOT_DIR, res_label, data_filename)

    # Check if input file exists to avoid runtime error
    if not os.path.isfile(data_path):
        raise FileNotFoundError(f"Data file not found: {data_path}")

    # -------------------------- 核心修改：读取 NPZ 格式 --------------------------
    # Load NPZ binary file & extract plotting data (key = 'value')
    npz_data = np.load(data_path)
    sfr_map = npz_data['value'].copy()

    # Mark empty pixels with healpy standard UNSEEN flag
    sfr_map[sfr_map == EMPTY_PIXEL_VALUE] = hp.UNSEEN

    # Calculate color scale limits
    valid_data = sfr_map[sfr_map != hp.UNSEEN]
    if len(valid_data) > 0:
        val_max = np.max(valid_data)
        val_min = np.min(valid_data)
        if val_max - COLOR_SCALE_OFFSET > val_min:
            val_min = val_max - COLOR_SCALE_OFFSET
    else:
        # Default range for empty maps
        val_max, val_min = 0, -3

    # Generate Mollweide all-sky projection
    projview(
        sfr_map,
        coord=['C'],
        flip='astro',
        projection_type='mollweide',
        graticule=True,
        graticule_labels=True,
        nest=False,
        unit=r'log$(\rm SFR/ M_{\odot}yr^{-1})$',
        xlabel='RA',
        ylabel='DEC',
        cb_orientation='horizontal',
        min=val_min,
        max=val_max,
        latitude_grid_spacing=GRID_SPACING,
        longitude_grid_spacing=GRID_SPACING,
        fontsize={
            'xlabel': 21, 'ylabel': 21,
            'xtick_label': 20, 'ytick_label': 30,
            'cbar_label': 30, 'cbar_tick_label': 30
        },
        width=FIG_WIDTH,
        cmap='binary',
        cbar=True
    )

    # Create output directory automatically
    fig_subdir = os.path.join(FIGURE_SAVE_DIR, res_label)
    os.makedirs(fig_subdir, exist_ok=True)

    # Save figure and close canvas to release memory
    fig_filename = f"SFR_D{distance_cut}_{res_label}.pdf"
    fig_save_path = os.path.join(fig_subdir, fig_filename)
    plt.savefig(fig_save_path, dpi=PLOT_DPI, format="pdf", bbox_inches='tight')
    plt.close()

    # Print formatted progress log
    print(
        f"Finished | Resolution={res_label:12s} | Distance={distance_cut:3d} Mpc "
        f"| Saved to: {fig_save_path}"
    )

# -------------------------- Main Program Entry --------------------------
def main():
    print("=" * 60)
    print("Start Batch Visualization for SFR HEALPix Maps")
    print("=" * 60)

    # Iterate over all target HEALPix resolutions
    for nside in TARGET_NSIDES:
        current_res = NSIDE_RES_MAPPING[nside]
        print(f"\n----- Processing Resolution: {current_res} (NSIDE = {nside}) -----")

        # Iterate over all distance thresholds
        for dist in DISTANCE_RANGE:
            plot_and_export_sfr_sky_map(current_res, dist)

    # Final completion log
    print("\n" + "=" * 60)
    print("All batch plotting tasks completed successfully.")
    print(f"All figures saved to: {FIGURE_SAVE_DIR}")
    print("=" * 60)

# Standard entry point: support standalone run & module import
if __name__ == "__main__":
    main()